In [2]:
import json
import yaml
import folium
import urllib3
import numpy as np
import pandas as pd
import seaborn as sns
import geopandas as gpd
from itertools import product
import matplotlib.pyplot as plt
from scipy.stats import bootstrap, ecdf
from shapely.geometry import Point, Polygon, MultiPolygon
import seaborn as sns
import warnings
warnings.resetwarnings()


### Open z-score and modified z-score file

In [2]:
save_out_path = "/glade/campaign/cisl/aiml/cat_flood/mrms/deltas/"
date_str = "20251006_2245"
file_path = f"veraset_users_mrms_{date_str}_stats_deltas.parquet"
df = pd.read_parquet(save_out_path+file_path, engine='pyarrow')

### Create boolean columns for FFWs and rain amounts during the AR period

#### FFWs

In [4]:
def get_bool(row, polys):
    point = Point(row[["lng_home", "lat_home"]])
    return any([p.contains(point) for p in polys])

In [5]:
AR_dates = pd.date_range("2024-03-29", "2024-04-02", freq="1d")

for i, date in enumerate(AR_dates[:-1]):
    url = f"https://mesonet.agron.iastate.edu/api/1/cow.json?wfo=LOX&begints={AR_dates[i].strftime("%Y-%m-%d") }T00:00:00Z&enable_shared_border=1&endts={AR_dates[i+1].strftime("%Y-%m-%d") }T00:00:00Z&hailsize=1&wind=58&phenomena=FF&lsrtype=FF&lsrbuffer=30&warningbuffer=1"
    req = urllib3.request("GET", url)
    ffw_warnings = json.loads(req.data)["events"]["features"]
    polygons = [MultiPolygon(w["geometry"]["coordinates"]) for w in ffw_warnings]
    df[f"{date}_ffw"] = df.apply(get_bool, polys=polygons, axis=1)
    df[f"{date}_ffw"] = df[f"{date}_ffw"].replace({True: 'yes', False: 'no'})
df[[f"{d}_ffw" for d in AR_dates[:-1]]].describe()

,2024-03-29_ffw,2024-03-30_ffw,2024-03-31_ffw,2024-04-01_ffw
count,471,471,471,471
unique,1,2,2,1
top,no,no,no,no
freq,471,268,468,471


#### MRMS Variables

In [7]:
date_strs = ["2024-03-15T00:00_2024-04-29T00:00", "2023-03-15T00:00_2023-04-29T00:00"]
all_dates = pd.date_range(date_strs[0].split("_")[0], date_strs[0].split("_")[1], freq="1d").union(pd.date_range(date_strs[1].split("_")[0], date_strs[1].split("_")[1], freq="1d"))

for mrms_metric in ['precip_max', 'precip_sum', 'streamflow_max', 'streamflow_sum']:
    for date in AR_dates:
        idx = all_dates.get_loc(date)
        df[f"{mrms_metric}_{date.strftime('%Y-%m-%d')}"] = df.apply(lambda x: "yes" if x[mrms_metric][idx] > 0 else "no", axis=1)

### Calculate Bootsrap average, sign, and p-value

In [8]:
for mrms_label in ["precip", "streamflow"]:
    mrms_metrics = [c for c in list(df.columns) if any(k in c for k in [f'{mrms_label}_max_', f'{mrms_label}_sum_'])]
    mrms_metrics_bool = [c+b for c in mrms_metrics for b in ["_yes", "_no"]]
    mrms_metrics_bool_bs = [c+bs for c in mrms_metrics_bool for bs in ["_pvalue", "_boot_mean", "_boot_sign"]]
    deltas = [c for c in list(df.columns) if any(k in c for k in ['_z', '_modz'])]
    counties = ["LA", "SB", "VE"]
    bools = ["yes", "no"]
    
    for county in counties:
        df_bs = pd.DataFrame(columns=mrms_metrics_bool_bs, index=deltas)
        for mrms_metric in mrms_metrics:
            for delta in deltas:
                for bool in bools:
                    data = df.loc[(df[mrms_metric]==bool) & (df['county']==county)][delta].dropna()
                    if data.shape[0]<2:
                        df_bs.loc[delta, f"{mrms_metric}_{bool}_pvalue"] = np.nan
                        continue
                    else:
                        res = bootstrap((data,), np.mean, confidence_level=0.95, paired=False, alternative='two-sided')
                        res_ecdf = ecdf(res.bootstrap_distribution)
                        res_ecdf_0 = float(res_ecdf.cdf.evaluate(0))
                        if res_ecdf_0 > 0.5:
                            df_bs.loc[delta, f"{mrms_metric}_{bool}_pvalue"] = 1.0 - res_ecdf_0
                        else:
                            df_bs.loc[delta, f"{mrms_metric}_{bool}_pvalue"] = res_ecdf_0
                        df_bs.loc[delta, f"{mrms_metric}_{bool}_boot_mean"] = np.mean(res.bootstrap_distribution)
                        df_bs.loc[delta, f"{mrms_metric}_{bool}_boot_sign"] = np.sign(np.mean(res.bootstrap_distribution))
        df_bs.to_parquet(f"bootstrapping_daily/results_{mrms_label}_{county}.parquet")

In [10]:
mrms_metrics = [c for c in list(df.columns) if "_ffw" in c]
mrms_metrics_bool = [c+b for c in mrms_metrics for b in ["_yes", "_no"]]
mrms_metrics_bool_bs = [c+bs for c in mrms_metrics_bool for bs in ["_pvalue", "_boot_mean", "_boot_sign"]]
deltas = [c for c in list(df.columns) if any(k in c for k in ['_z', '_modz'])]
counties = ["LA", "SB", "VE"]
bools = ["yes", "no"]

for county in counties:
    df_bs = pd.DataFrame(columns=mrms_metrics_bool_bs, index=deltas)
    for mrms_metric in mrms_metrics:
        for delta in deltas:
            for bool in bools:
                data = df.loc[(df[mrms_metric]==bool) & (df['county']==county)][delta].dropna()
                if data.shape[0]<2:
                    df_bs.loc[delta, f"{mrms_metric}_{bool}_pvalue"] = np.nan
                    continue
                else:
                    res = bootstrap((data,), np.mean, confidence_level=0.95, paired=False, alternative='two-sided')
                    res_ecdf = ecdf(res.bootstrap_distribution)
                    res_ecdf_0 = float(res_ecdf.cdf.evaluate(0))
                    if res_ecdf_0 > 0.5:
                        df_bs.loc[delta, f"{mrms_metric}_{bool}_pvalue"] = 1.0 - res_ecdf_0
                    else:
                        df_bs.loc[delta, f"{mrms_metric}_{bool}_pvalue"] = res_ecdf_0
                    df_bs.loc[delta, f"{mrms_metric}_{bool}_boot_mean"] = np.mean(res.bootstrap_distribution)
                    df_bs.loc[delta, f"{mrms_metric}_{bool}_boot_sign"] = np.sign(np.mean(res.bootstrap_distribution))
    df_bs.to_parquet(f"bootstrapping_daily/results_ffw_{county}.parquet")